# SNNAI Kaggle Setup

This notebook installs the SNNAI dependencies and verifies that the core SNN frameworks are available on Kaggle.

In [ ]:
# Install pinned dependencies (CPU/GPU compatible)
!pip install -q numpy==1.26.4 torch==2.3.0
!pip install -q snntorch==0.9.4 brian2==2.7.0 norse==1.1.0 bindsnet==0.3.3
!pip install -q matplotlib==3.9.0 pytest==8.2.0

In [ ]:
# Verify all imports and GPU availability
import sys
sys.path.append('/kaggle/working')

modules = ['torch', 'snntorch', 'brian2', 'norse', 'bindsnet', 'numpy', 'matplotlib', 'pytest']
for m in modules:
    try:
        __import__(m)
        print(f'[OK] {m}')
    except Exception as e:
        print(f'[NG] {m}: {e}')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

In [ ]:
# Single LIF neuron spike with snnTorch
import torch
import snntorch as snn

lif = snn.Leaky(beta=0.9, threshold=1.0)
mem = lif.init_leaky()
spikes = []
for _ in range(20):
    spk, mem = lif(torch.ones(1) * 1.5, mem)
    spikes.append(spk.item())
print('snnTorch LIF spikes:', spikes)
print('Total spikes:', sum(spikes))

In [ ]:
# Single LIF neuron spike with Brian2
from brian2 import *

eqs = '''\ndv/dt = (I - v) / tau : 1\nI : 1\ntau : second\n'''
G = NeuronGroup(1, eqs, threshold='v > 1.0', reset='v = 0', method='exact')
G.I = 1.5
G.tau = 10*ms
M = SpikeMonitor(G)
run(200*ms)
print('Brian2 spike times:', M.t[:])
print('Total spikes:', M.num_spikes)